<a href="https://colab.research.google.com/github/Chetan-Kumar-G/Machine-Learning/blob/main/ML_Lab_2_%E2%80%93_Logistic_Regression_for_GRE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
chetankumarg_graduate_record_exam_scores_path = kagglehub.dataset_download('chetankumarg/graduate-record-exam-scores')

print('Data source import complete.')


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# Machine Learning Lab – Experiment 2
## Logistic Regression – Graduate Admission Prediction

### Aim
To build a Logistic Regression model to predict whether a student
will be admitted to graduate school using GRE score, GPA, and
University Prestige.

In [ ]:
# ============================================
# 1. IMPORT REQUIRED LIBRARIES
# ============================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid")

print("Libraries imported successfully!")


In [ ]:
# ============================================
# 2. FIND THE DATASET
# ============================================

import os

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        print(os.path.join(root, file))

In [ ]:
# ============================================
# 3. LOAD DATASET
# ============================================

df = pd.read_stata("/kaggle/input/datasets/chetankumarg/graduate-record-exam-scores/binary.dta")

print("Dataset loaded successfully!")
print("Dataset Shape:", df.shape)

display(df.head())

In [ ]:
# ============================================
# 4. DATASET INFORMATION
# ============================================

print("Dataset Shape:", df.shape)

print("\nColumn Names:")
print(df.columns.tolist())

print("\nDataset Information:")
df.info()

In [ ]:
# ============================================
# 5. STATISTICAL SUMMARY
# ============================================

display(df.describe(include="all").T)

In [ ]:
# ============================================
# 6. MISSING VALUE ANALYSIS
# ============================================

missing_values = df.isnull().sum()

missing_summary = pd.DataFrame({
    "Missing Values": missing_values,
    "Percentage": (missing_values / len(df)) * 100
})

display(missing_summary)

In [ ]:
# ============================================
# 7. UNIQUE VALUES
# ============================================

for column in df.columns:
    print(f"\n{column}:")
    print(df[column].unique()[:20])

In [ ]:
# ============================================
# 8. ADMISSION DISTRIBUTION
# ============================================

plt.figure(figsize=(7, 5))

sns.countplot(data=df, x="admit")

plt.title("Graduate Admission Distribution")
plt.xlabel("Admission")
plt.ylabel("Number of Students")

plt.show()

In [ ]:
# ============================================
# 9. GRE SCORE VS ADMISSION
# ============================================

plt.figure(figsize=(8, 6))

sns.boxplot(
    data=df,
    x="admit",
    y="gre"
)

plt.title("GRE Score vs Admission")
plt.xlabel("Admission")
plt.ylabel("GRE Score")

plt.show()

In [ ]:
# ============================================
# 10. GPA VS ADMISSION
# ============================================

plt.figure(figsize=(8, 6))

sns.boxplot(
    data=df,
    x="admit",
    y="gpa"
)

plt.title("GPA vs Admission")
plt.xlabel("Admission")
plt.ylabel("GPA")

plt.show()

In [ ]:
# ============================================
# 11. UNIVERSITY RANK VS ADMISSION
# ============================================

plt.figure(figsize=(8, 6))

sns.countplot(
    data=df,
    x="rank",
    hue="admit"
)

plt.title("University Rank vs Admission")
plt.xlabel("University Rank")
plt.ylabel("Number of Students")

plt.show()

In [ ]:
# ============================================
# 12. DEFINE FEATURES AND TARGET
# ============================================

X = df[["gre", "gpa", "rank"]]
y = df["admit"]

print("Features:")
display(X.head())

print("\nTarget:")
display(y.head())

In [ ]:
# ============================================
# 13. HANDLE MISSING VALUES
# ============================================

print("Missing values before cleaning:")
print(df[["gre", "gpa", "rank", "admit"]].isnull().sum())

model_data = df[["gre", "gpa", "rank", "admit"]].dropna()

X = model_data[["gre", "gpa", "rank"]]
y = model_data["admit"]

print("\nDataset shape after cleaning:", model_data.shape)

In [ ]:
# ============================================
# 14. TRAIN-TEST SPLIT
# ============================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples :", X_test.shape[0])


### Feature Scaling

The three input variables are measured on different scales. I am standardizing them before training so that each feature is given a comparable scale. This helps the Logistic Regression model learn the relationship between the inputs more effectively.

In [ ]:
# ============================================
# 15. TRAIN LOGISTIC REGRESSION MODEL
# ============================================

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

model.fit(X_train_scaled, y_train)

print("Logistic Regression model trained successfully!")


In [ ]:
# ============================================
# 16. MAKE PREDICTIONS
# ============================================

y_pred = model.predict(X_test_scaled)

results = pd.DataFrame({
    "Actual Admission": y_test.values,
    "Predicted Admission": y_pred
})

display(results.head(10))


In [ ]:
# ============================================
# 17. PREDICTION PROBABILITIES
# ============================================

probabilities = model.predict_proba(X_test_scaled)

probability_results = pd.DataFrame({
    "Actual Admission": y_test.values,
    "Probability of Not Admitted": probabilities[:, 0],
    "Probability of Admitted": probabilities[:, 1],
    "Predicted Admission": y_pred
})

display(probability_results.head(10))


In [ ]:
# ============================================
# 18. CONFUSION MATRIX
# ============================================

cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:")
print(cm)

plt.figure(figsize=(7, 5))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=["Not Admitted", "Admitted"],
    yticklabels=["Not Admitted", "Admitted"]
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")

plt.show()


In [ ]:
# ============================================
# 19. MODEL EVALUATION
# ============================================

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

print("=" * 45)
print("       LOGISTIC REGRESSION RESULTS")
print("=" * 45)

print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 Score  : {f1:.4f}")

print("=" * 45)


In [ ]:
# ============================================
# 20. CLASSIFICATION REPORT
# ============================================

print(
    classification_report(
        y_test,
        y_pred,
        target_names=["Not Admitted", "Admitted"],
        zero_division=0
    )
)


In [ ]:
# ============================================
# 21. MODEL COEFFICIENTS
# ============================================

coefficients = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model.coef_[0]
})

display(coefficients)

print("Intercept:", model.intercept_[0])


In [ ]:
# ============================================
# 22. ACTUAL VS PREDICTED
# ============================================

comparison = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": y_pred
})

comparison = comparison.reset_index(drop=True)

display(comparison.head(20))

## Conclusion

A Logistic Regression model was successfully developed to predict
graduate-school admission using GRE score, GPA, and university rank.

The provided Stata (.dta) dataset was loaded and explored. The relevant
features, namely GRE score, GPA, and university rank, were selected as
independent variables, while admission status was used as the target
variable.

The dataset was divided into training and testing sets, and a Logistic
Regression model was trained on the training data.

The model was evaluated using a confusion matrix, Accuracy, Precision,
Recall, and F1 Score.

Thus, Logistic Regression was successfully applied to predict whether
a student would be admitted to graduate school.